# GEARS — Comparison with CHARME Simulation Examples

Loads an external time series (e.g. `CHARME.csv`) and compares it to the charging profiles reconstructed by GEARS.

**Usage:** adjust the constants in the configuration cell, then run all cells (`Kernel › Restart & Run All`).

## 1. Configuration

In [ ]:
# === CONFIGURATION — adjust to your use case ===
EXTERNAL_FILE  = "CharMEMT_SimMT_CharME20182025_NbV20182025_03032026.csv"      # path to the external simulation (relative to this notebook)
YEAR           = 2025            # year to analyze
N_DAYS_MC      = 10              # Monte-Carlo days to stabilize the GEARS profiles
CHARGING_MODE  = "fixed_power"   # "mean_power" | "fixed_power" | "by_location"
FIXED_POWER_KW = 7.4             # used only if CHARGING_MODE == "fixed_power"
RESOLUTION_MIN = 60              # time resolution for smart charging (minutes)
SCALE_TO_MATCH = True            # rescale GEARS to the external series' median

# Shared palette (external / GEARS)
COLOR_EXT   = "#2563EB"  # blue
COLOR_GEARS = "#F97316"  # orange


In [ ]:
import warnings

warnings.filterwarnings('ignore')

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import gaussian_kde

from gears import NativeSessionModelRegistry
from gears.data.schemas import _season
from gears.output.aggregator import OutputAggregator

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
})

# Short weekday names (0 = Monday)
DAY_NAMES = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
SEASON_ORDER = ["Winter", "Spring", "Summer", "Autumn"]
SEASON_LABELS = {"winter": "Winter", "spring": "Spring", "summer": "Summer", "autumn": "Autumn"}


## 2. Loading the External Simulation

Robust reading of `CHARME.csv`: `;` separator, comma decimal, UTC `Instant` column. Filters on `YEAR` and renames to standard columns.

In [ ]:
# Read the external file
csv_path = Path(EXTERNAL_FILE)
if not csv_path.is_absolute():
    csv_path = Path(__file__).parent / csv_path if '__file__' in dir() else Path(csv_path)

df_ext_raw = pd.read_csv(
    csv_path,
    sep=';',
    decimal=',',
    parse_dates=['Instant'],
)

# Drop the UTC timezone to simplify comparisons
df_ext_raw['Instant'] = df_ext_raw['Instant'].dt.tz_localize(None)

# Filter to the target year and rename
df_ext = (
    df_ext_raw[df_ext_raw['Instant'].dt.year == YEAR]
    .rename(columns={'Instant': 'time', 'P_MW': 'power_mw'})
    .set_index('time')
    [['power_mw']]
    .copy()
)

# Add calendar columns useful for the analysis
df_ext['hour']        = df_ext.index.hour
df_ext['dayofweek']   = df_ext.index.dayofweek  # 0=Monday
df_ext['day']         = df_ext['dayofweek'].map(lambda d: DAY_NAMES[d])
df_ext['is_weekend']  = df_ext['dayofweek'] >= 5
df_ext['month']       = df_ext.index.month
df_ext['season']      = df_ext['month'].map(_season).map(SEASON_LABELS)
df_ext['day_type']    = df_ext['is_weekend'].map({False: 'Weekday', True: 'Weekend'})

print(f"External series: {len(df_ext)} hours ({YEAR})")
print(f"  Range  : {df_ext.index.min()} → {df_ext.index.max()}")
print(f"  Mean   : {df_ext['power_mw'].mean():.1f} MW")
print(f"  Median : {df_ext['power_mw'].median():.1f} MW")
print(f"  Max    : {df_ext['power_mw'].max():.1f} MW")


## 3. Loading the GMM and Reconstructing the GEARS Profiles

Loads the `"french"` bundle from `NativeSessionModelRegistry`, then reconstructs an annual hourly profile via `OutputAggregator.build_load_profiles()`.

In [ ]:
registry = NativeSessionModelRegistry()
gmm = registry.load("french")
agg = OutputAggregator(resolution_min=RESOLUTION_MIN)

print(f"GMM loaded: {len(gmm.models_)} strata "
      f"({', '.join(gmm.stratify_by)})")


In [ ]:
result = agg.build_load_profiles(
    gmm=gmm,
    year=YEAR,
    n_days_mc=N_DAYS_MC,
    charging_mode=CHARGING_MODE,
    charger_power_kw=FIXED_POWER_KW,   # ignored if mode != 'fixed_power'
)

g_ts = result['ts'].copy()  # hourly pd.Series in MW

print(f"GEARS profiles built: mode={result['charging_mode']}")
print(f"  Raw mean   : {g_ts.mean():.3f} MW")
print(f"  Raw median : {g_ts.median():.3f} MW")
print(f"  Raw max    : {g_ts.max():.3f} MW")


In [ ]:
# ── Optional rescaling (median-to-median) ──────────────────────────────
if SCALE_TO_MATCH:
    ext_median   = df_ext['power_mw'].median()
    gears_median = g_ts.median()
    scale_factor = ext_median / gears_median
    g_ts = g_ts * scale_factor
    print(f"Rescaling enabled: ×{scale_factor:.4f} "
          f"(external median={ext_median:.1f} MW, GEARS raw={gears_median:.3f} MW)")
else:
    scale_factor = 1.0
    print("Rescaling disabled — both series are compared as-is.")

# Align indices: keep the intersection of timestamps
g_ts.index = pd.date_range(f"{YEAR}-01-01", periods=len(g_ts), freq='h')
common_idx  = df_ext.index.intersection(g_ts.index)
ext_aligned = df_ext.loc[common_idx, 'power_mw']
g_aligned   = g_ts.loc[common_idx]

# Calendar columns on the aligned GEARS series
g_df = g_aligned.rename('power_mw').to_frame()
g_df['hour']      = g_df.index.hour
g_df['dayofweek'] = g_df.index.dayofweek
g_df['day']       = g_df['dayofweek'].map(lambda d: DAY_NAMES[d])
g_df['is_weekend']= g_df['dayofweek'] >= 5
g_df['month']     = g_df.index.month
g_df['season']    = g_df['month'].map(_season).map(SEASON_LABELS)
g_df['day_type']  = g_df['is_weekend'].map({False: 'Weekday', True: 'Weekend'})

print(f"\nOverlapping points: {len(common_idx)} hours")


## 4. Smart Charging (optional)

Uncomment the block below to enable V1G optimization. A synthetic price signal is provided as an example — replace it with a real signal (e.g. hourly EPEX SPOT price) for realistic results.

In [ ]:
# ── Uncomment to enable smart charging ───────────────────────────────────
# import pandas as pd, numpy as np
#
# # Synthetic price signal: daily sinusoid + noise (€/kWh)
# # Trough at night (3am), peak in the evening (7pm) — EXAMPLE ONLY
# price_idx = pd.date_range(f"{YEAR}-01-01", periods=8760, freq='h')
# hours      = np.arange(8760) % 24
# price_signal = pd.Series(
#     0.12 + 0.04 * np.sin(2 * np.pi * (hours - 3) / 24)
#     + np.random.default_rng(42).normal(0, 0.005, 8760),
#     index=price_idx,
#     name='price_eur_per_kwh',
# )
#
# result_smart = agg.build_load_profiles(
#     gmm=gmm,
#     year=YEAR,
#     n_days_mc=N_DAYS_MC,
#     charging_mode=CHARGING_MODE,
#     charger_power_kw=FIXED_POWER_KW,
#     smart_charging_signal=price_signal,
# )
# g_ts_smart = result_smart['ts_smart'] * scale_factor  # same rescaling
# print(f"Smart-charging series built: "
#       f"mean={g_ts_smart.mean():.3f} MW, max={g_ts_smart.max():.3f} MW")

# Flag for the following cells
g_ts_smart = None  # update if the block above is run


## 5. Annual Time Series

Full-year overview — weekly smoothing for readability.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

# Raw data (transparent)
axes[0].plot(ext_aligned.index, ext_aligned.values,
             color=COLOR_EXT, alpha=0.25, lw=0.6, label='_nolegend_')
axes[0].plot(g_aligned.index, g_aligned.values,
             color=COLOR_GEARS, alpha=0.25, lw=0.6, label='_nolegend_')
# 7-day smoothing
axes[0].plot(ext_aligned.rolling(168, center=True).mean(),
             color=COLOR_EXT, lw=2, label='External (7d avg)')
axes[0].plot(g_aligned.rolling(168, center=True).mean(),
             color=COLOR_GEARS, lw=2, label='GEARS (7d avg)')
axes[0].set_ylabel('Power (MW)')
axes[0].set_title(f'Annual Time Series {YEAR} — 7-day Smoothing')
axes[0].legend(loc='upper left')

# Residual (difference)
residual = g_aligned.values - ext_aligned.values
axes[1].axhline(0, color='black', lw=0.8)
axes[1].plot(ext_aligned.index, pd.Series(residual, index=ext_aligned.index)
             .rolling(168, center=True).mean(),
             color='#7C3AED', lw=1.5, label='GEARS − External (7d avg)')
axes[1].fill_between(ext_aligned.index,
                     pd.Series(residual, index=ext_aligned.index)
                     .rolling(168, center=True).mean(),
                     0, alpha=0.15, color='#7C3AED')
axes[1].set_ylabel('Residual (MW)')
axes[1].set_title('Residual: GEARS − External')
axes[1].legend(loc='upper left')

fig.autofmt_xdate()
plt.tight_layout()
plt.show()


## 6. Power Distribution — Overall Density (KDE)

Comparison of the distribution shapes across the full year.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

for vals, label, color in [
    (ext_aligned.values, 'External', COLOR_EXT),
    (g_aligned.values,   'GEARS',    COLOR_GEARS),
]:
    kde = gaussian_kde(vals, bw_method='scott')
    x   = np.linspace(vals.min(), vals.max(), 400)
    ax.plot(x, kde(x), color=color, lw=2.5, label=label)
    ax.fill_between(x, kde(x), alpha=0.12, color=color)

ax.set_xlabel('Power (MW)')
ax.set_ylabel('Density')
ax.set_title('Overall Charging Power Distribution')
ax.legend()
plt.tight_layout()
plt.show()


## 7. Distribution by Day Type — Weekday vs Weekend

Checks that GEARS correctly reproduces the behavioral contrast between weekdays and weekends.

In [ ]:
day_types = ['Weekday', 'Weekend']
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)

for ax, dt in zip(axes, day_types):
    for df_src, label, color in [
        (df_ext,  'External', COLOR_EXT),
        (g_df,    'GEARS',    COLOR_GEARS),
    ]:
        vals = df_src.loc[df_src['day_type'] == dt, 'power_mw'].values
        if len(vals) < 10:
            continue
        kde = gaussian_kde(vals, bw_method='scott')
        x   = np.linspace(vals.min(), vals.max(), 300)
        ax.plot(x, kde(x), color=color, lw=2, label=label)
        ax.fill_between(x, kde(x), alpha=0.12, color=color)
    ax.set_title(dt)
    ax.set_xlabel('Power (MW)')
    ax.set_ylabel('Density')
    ax.legend()

fig.suptitle('Power Distribution — Weekday vs Weekend', fontsize=13)
plt.tight_layout()
plt.show()


## 8. Distribution by Season

Identifies any seasonal biases.

In [ ]:
season_order = ['Winter', 'Spring', 'Summer', 'Autumn']
palette_s    = ['#3B82F6', '#22C55E', '#F59E0B', '#EF4444']

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, (df_src, label) in zip(
    axes,
    [(df_ext, 'External'), (g_df, 'GEARS')],
):
    for season, color in zip(season_order, palette_s):
        vals = df_src.loc[df_src['season'] == season, 'power_mw'].values
        if len(vals) < 10:
            continue
        kde = gaussian_kde(vals, bw_method='scott')
        x   = np.linspace(vals.min(), vals.max(), 300)
        ax.plot(x, kde(x), color=color, lw=2, label=season)
        ax.fill_between(x, kde(x), alpha=0.08, color=color)
    ax.set_title(label)
    ax.set_xlabel('Power (MW)')
    ax.set_ylabel('Density')
    ax.legend(title='Season')

fig.suptitle('Power Distribution by Season', fontsize=13)
plt.tight_layout()
plt.show()


## 9. Average Hourly Profile by Day of Week

7 overlaid curves (Mon–Sun) — external on the left, GEARS on the right. Lets you check the intra-day shape and peaks.

In [ ]:
palette_dow = plt.cm.tab10(np.linspace(0, 0.7, 7))

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

for ax, (df_src, label) in zip(
    axes,
    [(df_ext, 'External'), (g_df, 'GEARS')],
):
    for dow_i, (day, color) in enumerate(zip(DAY_NAMES, palette_dow)):
        mask    = df_src['dayofweek'] == dow_i
        profile = df_src.loc[mask].groupby('hour')['power_mw'].mean()
        ax.plot(profile.index, profile.values, color=color, lw=2, label=day)
    ax.set_xlabel('Hour of Day')
    ax.set_ylabel('Average Power (MW)')
    ax.set_title(label)
    ax.set_xticks(range(0, 24, 3))
    ax.legend(title='Day', ncol=2, fontsize=8)

fig.suptitle('Average Hourly Profile by Day of Week', fontsize=13)
plt.tight_layout()
plt.show()


## 10. Average Hourly Profile by Season

4 curves — external on the left, GEARS on the right.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

for ax, (df_src, label) in zip(
    axes,
    [(df_ext, 'External'), (g_df, 'GEARS')],
):
    for season, color in zip(season_order, palette_s):
        mask    = df_src['season'] == season
        profile = df_src.loc[mask].groupby('hour')['power_mw'].mean()
        ax.plot(profile.index, profile.values, color=color, lw=2.5, label=season)
    ax.set_xlabel('Hour of Day')
    ax.set_ylabel('Average Power (MW)')
    ax.set_title(label)
    ax.set_xticks(range(0, 24, 3))
    ax.legend(title='Season')

fig.suptitle('Average Hourly Profile by Season', fontsize=13)
plt.tight_layout()
plt.show()


## 11. Average Energy by Day of Week

Comparative barplot — one bar per source and per day. Daily energy is the sum of hourly power values (MW·h ≈ MWh).

In [ ]:
# Compute daily energy (sum of MW over 24h = MWh)
def energy_by_dow(df_src):
    daily = (
        df_src.assign(date=df_src.index.normalize())
        .groupby(['date', 'dayofweek'])['power_mw'].sum()
        .reset_index()
    )
    return daily.groupby('dayofweek')['power_mw'].mean()

eng_ext   = energy_by_dow(df_ext)
eng_gears = energy_by_dow(g_df)

x   = np.arange(7)
w   = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
bars_ext  = ax.bar(x - w/2, eng_ext.values,   w, color=COLOR_EXT,   label='External', alpha=0.85)
bars_gear = ax.bar(x + w/2, eng_gears.values, w, color=COLOR_GEARS, label='GEARS',    alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(DAY_NAMES)
ax.set_xlabel('Day of Week')
ax.set_ylabel('Average Energy (MWh)')
ax.set_title('Average Daily Charging Energy by Day of Week')
ax.legend()
plt.tight_layout()
plt.show()


## 12. Plug-and-Charge vs Smart Charging Comparison (if enabled)

This section is only active if `g_ts_smart` was computed in step 4. It compares hourly profiles over a representative week.

In [ ]:
if g_ts_smart is not None:
    # Representative week: first full week of March
    week_start = pd.Timestamp(f'{YEAR}-03-03')  # Monday
    week_end   = week_start + pd.Timedelta(days=6, hours=23)

    ts_plug_week  = g_aligned.loc[week_start:week_end]
    ts_smart_week = g_ts_smart.loc[week_start:week_end]

    fig, ax = plt.subplots(figsize=(14, 5))
    ax.plot(ts_plug_week.index, ts_plug_week.values,
            color=COLOR_GEARS, lw=2, label='GEARS — Plug-and-charge', alpha=0.8)
    ax.plot(ts_smart_week.index, ts_smart_week.values,
            color='#059669', lw=2, linestyle='--', label='GEARS — Smart charging (V1G)')
    ax.set_xlabel('Date / Time')
    ax.set_ylabel('Power (MW)')
    ax.set_title('Charging Profile — Week of March 3rd (plug-and-charge vs smart charging)')
    ax.legend()
    fig.autofmt_xdate()
    plt.tight_layout()
    plt.show()
else:
    print("Smart charging not enabled. Uncomment cell 4 to enable it.")


## 13. Summary Metrics

Summary table for a quick comparison of the two sources.

In [ ]:
def peak_hour(series):
    """Modal hour of the daily peak (median over the year)."""
    daily_peaks = (
        series.to_frame('mw')
        .assign(date=series.index.normalize(),
                hour=series.index.hour)
        .loc[series.to_frame('mw')
             .assign(date=series.index.normalize(),
                     hour=series.index.hour)
             .groupby('date')['mw'].transform('max')
             == series.to_frame('mw').assign(date=series.index.normalize())['mw']]
        ['hour']
    )
    return int(daily_peaks.mode()[0])

def annual_energy_twh(series):
    """Sum of hourly MW → TWh (MW·h / 1,000,000)."""
    return series.sum() / 1_000_000

metrics = {
    'Metric': [
        'Mean (MW)',
        'Median (MW)',
        'Max (MW)',
        'Peak hour (hour)',
        'Annual energy (TWh)',
    ],
    'External': [
        f"{ext_aligned.mean():.2f}",
        f"{ext_aligned.median():.2f}",
        f"{ext_aligned.max():.2f}",
        f"{peak_hour(ext_aligned):02d}h",
        f"{annual_energy_twh(ext_aligned):.4f}",
    ],
    'GEARS': [
        f"{g_aligned.mean():.2f}",
        f"{g_aligned.median():.2f}",
        f"{g_aligned.max():.2f}",
        f"{peak_hour(g_aligned):02d}h",
        f"{annual_energy_twh(g_aligned):.4f}",
    ],
}

df_metrics = pd.DataFrame(metrics)
print(df_metrics.to_string(index=False))

# Relative differences
print("\n— Relative bias, GEARS vs External —")
for k, ext_v, g_v in zip(
    metrics['Metric'][:3],
    [ext_aligned.mean(), ext_aligned.median(), ext_aligned.max()],
    [g_aligned.mean(),   g_aligned.median(),   g_aligned.max()],
):
    bias = (g_v - ext_v) / ext_v * 100
    print(f"  {k:<25s}: {bias:+.1f}%")
